# AI-Powered Paragraph Analyzer  
### *Sentiment • Keywords • Summary • Generative AI 

---

In [1]:
import nltk
import os
import ssl
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

In [2]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [3]:
# Set explicit NLTK data path (current directory + 'nltk_data')
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
if nltk_data_path not in nltk.data.path:
    nltk.data.path.insert(0, nltk_data_path)


In [4]:
os.makedirs(nltk_data_path, exist_ok=True)


In [5]:

def download_nltk_data(resource, name):
    print(f" Downloading {name}...")
    try:
        nltk.download(resource, download_dir=nltk_data_path, quiet=False, raise_on_error=True)
        print(f" {name} downloaded successfully to {nltk_data_path}")
    except Exception as e:
        print(f" Failed to download {name}: {e}")
        raise

In [6]:

# Download required resources
download_nltk_data('punkt', 'punkt tokenizer')
download_nltk_data('punkt_tab', 'punkt_tab tokenizer')
download_nltk_data('stopwords', 'stopwords')
download_nltk_data('vader_lexicon', 'VADER lexicon')

 punkt tokenizer downloaded successfully to d:\SmartContent\nltk_data
 punkt_tab tokenizer downloaded successfully to d:\SmartContent\nltk_data
 stopwords downloaded successfully to d:\SmartContent\nltk_data
 VADER lexicon downloaded successfully to d:\SmartContent\nltk_data


[nltk_data] Downloading package punkt to d:\SmartContent\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     d:\SmartContent\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     d:\SmartContent\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     d:\SmartContent\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
print("Verifying downloads...")
try:
    # Test word_tokenize (uses punkt_tab)
    print("Tokenize test:", word_tokenize("Hello world, testing NLTK!"))
    
    # Test stopwords
    print("Stopwords sample:", stopwords.words('english')[:5])
    
    # Test VADER
    sia = SentimentIntensityAnalyzer()
    print("VADER test:", sia.polarity_scores("I love NLTK!")['compound'])
    
    print("All NLTK resources verified and working!")
except Exception as e:
    print(f"Verification failed: {e}")
    raise


Verifying downloads...
Tokenize test: ['Hello', 'world', ',', 'testing', 'NLTK', '!']
Stopwords sample: ['a', 'about', 'above', 'after', 'again']
VADER test: 0.6696
All NLTK resources verified and working!


In [8]:

import spacy
from transformers import pipeline
from collections import Counter
import string
import re

C:\Users\LENOVO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:

print(" Loading spaCy and Hugging Face models...")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("  en_core_web_sm not found. Installing...")
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

sia = SentimentIntensityAnalyzer()
stop_words = set(stopwords.words('english'))

generator = pipeline('text-generation', 
                     model='distilgpt2',
                     max_length=100,
                     do_sample=True,
                     temperature=0.7)

print(" All models loaded!")

 Loading spaCy and Hugging Face models...


Device set to use cpu


 All models loaded!


In [10]:
class ContentAnalyzer:
    def __init__(self):
        self.nlp = nlp
        self.sia = sia
        self.stop_words = stop_words
        
    def analyze_text(self, text):
        analysis = {}
        analysis['word_count'] = len(text.split())
        analysis['char_count'] = len(text)
        analysis['sentence_count'] = len(list(self.nlp(text).sents))
        
        sentiment_scores = self.sia.polarity_scores(text)
        analysis['sentiment'] = sentiment_scores
        
        doc = self.nlp(text)
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        analysis['entities'] = entities
        
        pos_tags = [token.pos_ for token in doc if not token.is_space]
        analysis['pos_distribution'] = dict(Counter(pos_tags))
        
        return analysis
    
    def extract_keywords(self, text, top_n=10):
        # Tokenize safely
        try:
            tokens = word_tokenize(text.lower())
        except LookupError:
            nltk.download('punkt', download_dir=nltk_data_path)
            tokens = word_tokenize(text.lower())
            
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token not in string.punctuation
                 and token.isalpha()]
        
        doc = self.nlp(text)
        noun_phrases = [chunk.text for chunk in doc.noun_chunks]
        
        word_freq = Counter(tokens)
        keywords = word_freq.most_common(top_n)
        
        return {
            'frequency_based': keywords,
            'noun_phrases': noun_phrases[:top_n]
        }
    
    def generate_summary(self, text, max_sentences=3):
        doc = self.nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
        
        scored_sentences = []
        for i, sent in enumerate(sentences):
            score = len(sent.split())
            sent_doc = self.nlp(sent)
            score += len(sent_doc.ents) * 2
            scored_sentences.append((sent, score, i))
        
        scored_sentences.sort(key=lambda x: x[1], reverse=True)
        top_sentences = sorted(scored_sentences[:max_sentences], key=lambda x: x[2])
        
        summary = ' '.join([sent[0] for sent in top_sentences])
        return summary


In [ ]:

def main():
    print("\n STARTING CONTENT ANALYZER...\n")
    
    sample_text = """
Business Intelligence (BI) refers to the technologies, processes, and tools that organizations use to collect, analyze, and present business data in a meaningful way. By transforming raw, unstructured data into actionable insights, BI enables companies to make informed, data-driven decisions that improve performance and competitiveness. The core components of a BI system typically include data warehouses, ETL pipelines, OLAP cubes, and interactive dashboards. Data warehouses serve as centralized repositories where data from multiple sources is stored, cleaned, and structured for analysis. ETL processes, which stand for Extract, Transform, and Load, are responsible for moving data from source systems into the warehouse while ensuring its quality and consistency. Tools such as Power BI, Tableau, and Apache Superset are widely used by analysts and decision-makers to visualize key performance indicators, monitor business trends, and generate comprehensive reports. Beyond traditional reporting, modern BI platforms increasingly integrate machine learning algorithms to deliver predictive analytics, allowing organizations to forecast future outcomes based on historical patterns. Prescriptive analytics takes this a step further by not only predicting what will happen but also recommending the best course of action. With the exponential growth of data in today's digital economy, Business Intelligence has become an essential strategic asset for organizations across all industries, from retail and finance to healthcare and logistics, enabling them to stay agile, reduce costs, and seize new market opportunities.  
    """
    
    analyzer = ContentAnalyzer()
    
    print(" ANALYZING TEXT...")
    analysis = analyzer.analyze_text(sample_text)
    
    print(" ANALYSIS RESULTS:")
    print(f"Word Count: {analysis['word_count']}")
    print(f"Character Count: {analysis['char_count']}")
    print(f"Sentence Count: {analysis['sentence_count']}")
    print(f"Sentiment: {analysis['sentiment']['compound']:.3f} (compound)")
    print(f"Entities: {[e[0] for e in analysis['entities'][:5]]}")
    print(f"Top POS: {dict(list(analysis['pos_distribution'].items())[:5])}")
    
    print("\n KEYWORDS...")
    keywords = analyzer.extract_keywords(sample_text)
    print(f"Top Words: {keywords['frequency_based'][:5]}")
    print(f"Noun Phrases: {keywords['noun_phrases'][:5]}")
    
    print("\n SUMMARY...")
    summary = analyzer.generate_summary(sample_text)
    print(f"Summary: {summary}")
    
    print("\n GENERATIVE AI...")
    try:
        result = generator("The future of AI in healthcare:", max_length=80, num_return_sequences=1,truncation=True)
        print(f"Generated: {result[0]['generated_text']}")
    except Exception as e:
        print(f"  Generator failed: {e}")

# ========================
# 7. RUN
# ========================

if __name__ == "__main__":
    main()


 STARTING CONTENT ANALYZER...

 ANALYZING TEXT...


 ANALYSIS RESULTS:
Word Count: 1360
Character Count: 9538
Sentence Count: 65
Sentiment: 1.000 (compound)
Entities: ['Artificial Intelligence (AI', 'AI', 'ML', 'NLP', 'AI']
Top POS: {'PROPN': 92, 'PUNCT': 245, 'VERB': 172, 'PART': 30, 'ADP': 160}

 KEYWORDS...
Top Words: [('ai', 47), ('healthcare', 36), ('patient', 14), ('data', 13), ('treatment', 10)]
Noun Phrases: ['\nArtificial Intelligence', 'AI', 'healthcare', 'a transformative era', 'diagnostics']

 SUMMARY...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Summary: It examines the uses and effects of AI on healthcare by synthesizing recent literature and real‐world case studies, such as Google Health and IBM Watson Health, highlighting AI technologies, their useful applications, and the difficulties in putting them into practice, including problems with data security and resource limitations. Results
The findings demonstrate how AI is enhancing the skills of medical professionals, enhancing diagnosis, and opening the door to more individualized treatment plans, as reflected in the steady rise of AI‐related healthcare publications from 158 articles (3.54%) in 2014 to 731 articles (16.33%) by 2024. This in‐depth study looks at how AI is significantly impacting the healthcare sector, improving diagnostic precision through data analysis, streamlining treatment planning through predictive algorithms, and shedding light on how these advancements are challenging accepted wisdom and setting new benchmarks for quality [3, 4, 5].

 GENERATIVE AI..